In [ ]:
!pip install fastapi uvicorn pydantic openai

In [2]:
from pydantic import BaseModel, Field, field_validator
import re

class QueryRequest(BaseModel):
    query: str = Field(
        ...,
        example="What is machine learning?"
    )

    @field_validator("query")
    @classmethod
    def validate_query(cls, value):

        if len(value.strip()) < 5:
            raise ValueError("INPUT_INVALID: Query too short")

        if len(value) > 1000:
            raise ValueError("INPUT_INVALID: Query too long")

        if not re.search(r"[a-zA-Z]", value):
            raise ValueError("INPUT_INVALID: Query contains no valid text")

        return value


class QueryResponse(BaseModel):
    answer: str
    sources: list
    confidence: float

/tmp/ipykernel_2282/558878885.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  query: str = Field(


In [3]:
from pydantic import BaseModel

class ErrorResponse(BaseModel):
    code: str
    message: str
    request_id: str

In [4]:
def rag_pipeline(query):

    if "retrieval_fail" in query:
        return None

    return {
        "answer": f"Answer for: {query}",
        "sources": ["doc1", "doc2"],
        "confidence": 0.85
    }

In [5]:
import time

def call_llm_with_retry(query):

    max_retries = 3
    wait_time = 1

    for attempt in range(max_retries):

        try:

            # Simulate LLM call
            return f"Generated answer for {query}"

        except Exception:

            if attempt == max_retries - 1:
                raise

            time.sleep(wait_time)
            wait_time *= 2

In [6]:
import asyncio

async def fake_stream():

    words = [
        "This ",
        "is ",
        "a ",
        "streaming ",
        "response."
    ]

    for word in words:
        yield word
        await asyncio.sleep(0.5)

In [7]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from fastapi import HTTPException
import uuid
import asyncio

app = FastAPI()

In [8]:
@app.post("/ask")
async def ask(request: QueryRequest):

    request_id = str(uuid.uuid4())

    try:

        result = await asyncio.wait_for(
            process_query(request.query),
            timeout=15
        )

        return result

    except asyncio.TimeoutError:

        return {
            "code": "LLM_TIMEOUT",
            "message": "Request exceeded 15 seconds",
            "request_id": request_id
        }

In [9]:
async def process_query(query):

    result = rag_pipeline(query)

    if result is None:

        return {
            "code": "RETRIEVAL_FAILURE",
            "message": "No relevant document found",
            "request_id": str(uuid.uuid4())
        }

    return result

In [10]:
@app.get("/stream")
async def stream():

    return StreamingResponse(
        fake_stream(),
        media_type="text/plain"
    )

In [11]:
!pip install nest_asyncio

In [12]:
import nest_asyncio
nest_asyncio.apply()

In [14]:
!pip install pyngrok

In [16]:
from pyngrok import ngrok

ngrok.set_auth_token("3ESFZfnS7fJtKJif8X0UZz8KBgC_7TjBtNRvzEy77CSpN9n5D")

In [17]:
public_url = ngrok.connect(8000)
print(public_url)

NgrokTunnel: "https://epidermal-wise-unaudited.ngrok-free.dev" -> "http://localhost:8000"


In [18]:
{
  "query":"hi"
}

{'query': 'hi'}

In [19]:
{
  "query":"retrieval_fail"
}

{'query': 'retrieval_fail'}

In [20]:
{
  "query":"timeout"
}

{'query': 'timeout'}